# Brain Tumor Classification

### Install Required Libraries

### Import Libraries

In [4]:
import os, random, copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_score, recall_score, f1_score)

# Use GPU if available, otherwise CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

# Fix random seed so results are reproducible
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)
print('Libraries loaded!')

Using device: cpu
Libraries loaded!


### Set Dataset Path

In [6]:
# Auto-detect the notebook folder
BASE       = os.getcwd()
TRAIN_PATH = os.path.join(BASE, 'archive_extracted', 'Training')
TEST_PATH  = os.path.join(BASE, 'archive_extracted', 'Testing')

print('Training folder:', TRAIN_PATH)
print('Testing  folder:', TEST_PATH)
print('Train exists:', os.path.exists(TRAIN_PATH))
print('Test  exists:', os.path.exists(TEST_PATH))

# Settings
IMG_SIZE    = 128     # resize all images to 128x128 pixels
BATCH_SIZE  = 32      # process 32 images at a time
NUM_CLASSES = 4       # glioma, meningioma, notumor, pituitary
EPOCHS      = 15      # how many times to train over the full dataset

Training folder: C:\Users\ARCHANA\computervision_cw\archive_extracted\Training
Testing  folder: C:\Users\ARCHANA\computervision_cw\archive_extracted\Testing
Train exists: True
Test  exists: True


#### Prepare Data Loaders

In [10]:
# --- Transforms for Model 1 & 2 (trained from scratch) ---

# Training: resize + random flips/rotations to make model see variety
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # make all images same size
    transforms.RandomHorizontalFlip(),         # randomly flip left-right
    transforms.RandomRotation(15),             # randomly rotate up to 15 degrees
    transforms.ToTensor(),                     # convert image to numbers (0-1)
    transforms.Normalize([0.5, 0.5, 0.5],     # normalise: centre values around 0
                         [0.5, 0.5, 0.5]),
])

# Testing: only resize and normalise (no random changes)
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

# Load images from folders automatically (folder name = class label)
train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)
test_dataset  = datasets.ImageFolder(TEST_PATH,  transform=test_transform)

# DataLoader feeds batches of images to the model during training
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = train_dataset.classes   # ['glioma', 'meningioma', 'notumor', 'pituitary']
print('Classes:', CLASS_NAMES)
print(f'Train: {len(train_dataset)} images  |  Test: {len(test_dataset)} images')

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Train: 5600 images  |  Test: 1600 images


#### Training & Evaluation Helper Functions

In [13]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Train the model for ONE epoch (one full pass through training data)."""
    model.train()   # put model in training mode
    total_loss = 0
    correct    = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()          # clear old gradients
        outputs = model(images)        # forward pass: get predictions
        loss    = criterion(outputs, labels)   # calculate how wrong we are
        loss.backward()                # backward pass: calculate gradients
        optimizer.step()               # update model weights

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct   / len(loader.dataset)
    return avg_loss, accuracy


def evaluate(model, loader, criterion):
    """Evaluate the model on test data (no weight updates)."""
    model.eval()    # put model in evaluation mode
    total_loss = 0
    correct    = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():   # don't calculate gradients (saves memory)
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds       = outputs.argmax(1)
            correct    += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct   / len(loader.dataset)
    return avg_loss, accuracy, all_preds, all_labels


def train_model(model, train_loader, test_loader, epochs, lr, name):
    """Full training loop: trains for N epochs and saves the best weights."""
    model     = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()                          # loss function
    optimizer = optim.Adam(model.parameters(), lr=lr)          # Adam optimiser
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)  # reduce LR every 5 epochs

    history    = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc   = 0.0
    best_wts   = copy.deepcopy(model.state_dict())

    print(f'\n--- Training: {name} ---')
    for epoch in range(1, epochs + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        v_loss, v_acc, _, _ = evaluate(model, test_loader, criterion)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        # Save weights if this is the best accuracy so far
        if v_acc > best_acc:
            best_acc = v_acc
            best_wts = copy.deepcopy(model.state_dict())

        print(f'  Epoch {epoch:02d}/{epochs}  '
              f'Train Loss: {t_loss:.4f}  Train Acc: {t_acc:.4f}  '
              f'Val Loss: {v_loss:.4f}  Val Acc: {v_acc:.4f}')

    model.load_state_dict(best_wts)   # restore best weights
    print(f' Best Val Accuracy: {best_acc:.4f}')
    return model, history


print(' Helper functions ready!')


 Helper functions ready!


#### Model 1-Baseline CNN (From Scratch)

In [29]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Feature extractor: 3 blocks of Conv → ReLU → MaxPool
        self.features = nn.Sequential(
            # Block 1: input 3 channels → 32 feature maps
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # image size: 128 → 64

            # Block 2: 32 → 64 feature maps
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # image size: 64 → 32

            # Block 3: 64 → 128 feature maps
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # image size: 32 → 16
        )

        # Classifier: flatten and use fully-connected layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Linear(256, 4),        # 4 output classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Train Model 1
model1, history1 = train_model(
    model      = BaselineCNN(),
    train_loader = train_loader,
    test_loader  = test_loader,
    epochs = EPOCHS,
    lr     = 0.001,
    name   = 'Baseline CNN'
)

torch.save(model1.state_dict(), 'model1_baseline.pth')
print('Model 1 saved!')



--- Training: Baseline CNN ---
  Epoch 01/15  Train Loss: 0.7122  Train Acc: 0.7191  Val Loss: 0.9851  Val Acc: 0.7019
  Epoch 02/15  Train Loss: 0.4167  Train Acc: 0.8364  Val Loss: 0.7313  Val Acc: 0.8106
  Epoch 03/15  Train Loss: 0.3059  Train Acc: 0.8782  Val Loss: 0.7148  Val Acc: 0.8438
  Epoch 04/15  Train Loss: 0.2248  Train Acc: 0.9150  Val Loss: 0.7810  Val Acc: 0.8681
  Epoch 05/15  Train Loss: 0.1796  Train Acc: 0.9348  Val Loss: 0.7368  Val Acc: 0.8638
  Epoch 06/15  Train Loss: 0.1097  Train Acc: 0.9593  Val Loss: 0.8972  Val Acc: 0.9181
  Epoch 07/15  Train Loss: 0.0869  Train Acc: 0.9677  Val Loss: 0.7758  Val Acc: 0.9206
  Epoch 08/15  Train Loss: 0.0658  Train Acc: 0.9771  Val Loss: 1.0535  Val Acc: 0.9250
  Epoch 09/15  Train Loss: 0.0598  Train Acc: 0.9814  Val Loss: 1.0525  Val Acc: 0.9206
  Epoch 10/15  Train Loss: 0.0484  Train Acc: 0.9827  Val Loss: 1.1008  Val Acc: 0.9331
  Epoch 11/15  Train Loss: 0.0305  Train Acc: 0.9907  Val Loss: 1.2381  Val Acc: 0.9344


MemoryError: 

#### Model 2 - Improved CNN (From Scratch with Tuining)

In [15]:
class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),           # 128 → 64
            nn.Dropout2d(0.1),         # randomly zero 10% of feature maps

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),           # 64 → 32
            nn.Dropout2d(0.2),

            # Block 3
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),           # 32 → 16
            nn.Dropout2d(0.3),

            # Block 4
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),           # 16 → 8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 4),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Train Model 2 — lower LR and weight decay for better regularisation
model2, history2 = train_model(
    model        = ImprovedCNN(),
    train_loader = train_loader,
    test_loader  = test_loader,
    epochs = 20,
    lr     = 0.0005,
    name   = 'Improved CNN'
)

torch.save(model2.state_dict(), 'model2_improved.pth')
print('Model 2 saved!')



--- Training: Improved CNN ---
  Epoch 01/20  Train Loss: 1.2070  Train Acc: 0.5471  Val Loss: 1.0221  Val Acc: 0.6469
  Epoch 02/20  Train Loss: 0.7115  Train Acc: 0.7155  Val Loss: 0.7361  Val Acc: 0.7375
  Epoch 03/20  Train Loss: 0.6000  Train Acc: 0.7655  Val Loss: 0.7852  Val Acc: 0.7512
  Epoch 04/20  Train Loss: 0.5443  Train Acc: 0.7898  Val Loss: 0.8673  Val Acc: 0.7569
  Epoch 05/20  Train Loss: 0.4984  Train Acc: 0.8096  Val Loss: 0.9298  Val Acc: 0.7681
  Epoch 06/20  Train Loss: 0.4492  Train Acc: 0.8305  Val Loss: 0.7027  Val Acc: 0.7738
  Epoch 07/20  Train Loss: 0.4133  Train Acc: 0.8405  Val Loss: 0.7646  Val Acc: 0.8013
  Epoch 08/20  Train Loss: 0.4021  Train Acc: 0.8480  Val Loss: 0.7054  Val Acc: 0.8187
  Epoch 09/20  Train Loss: 0.3685  Train Acc: 0.8611  Val Loss: 0.7900  Val Acc: 0.8119
  Epoch 10/20  Train Loss: 0.3612  Train Acc: 0.8629  Val Loss: 0.7095  Val Acc: 0.8331
  Epoch 11/20  Train Loss: 0.3152  Train Acc: 0.8755  Val Loss: 0.7530  Val Acc: 0.8337


#### Model 3 - Fine-Tuned MobileNetV2(Transfer Learning)

In [24]:
# Pre-trained models need ImageNet normalisation values
pretrain_train = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet mean
                         [0.229, 0.224, 0.225]),   # ImageNet std
])
pretrain_test = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

pt_train_ds = datasets.ImageFolder(TRAIN_PATH, transform=pretrain_train)
pt_test_ds  = datasets.ImageFolder(TEST_PATH,  transform=pretrain_test)
pt_train_loader = DataLoader(pt_train_ds, batch_size=32, shuffle=True)
pt_test_loader  = DataLoader(pt_test_ds,  batch_size=32, shuffle=False)


# Load MobileNetV2 with pre-trained weights
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze backbone — don't update these weights during training
for param in mobilenet.features.parameters():
    param.requires_grad = False

# Replace the final classifier with our own 4-class head
in_features = mobilenet.classifier[1].in_features
mobilenet.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Linear(256, 4),
)

# Train Model 3
model3, history3 = train_model(
    model        = mobilenet,
    train_loader = pt_train_loader,
    test_loader  = pt_test_loader,
    epochs = EPOCHS,
    lr     = 0.001,
    name   = 'MobileNetV2 (Fine-tuned)'
)

torch.save(model3.state_dict(), 'model3_mobilenet.pth')
print('Model 3 saved!')

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\ARCHANA/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:05<00:00, 2.37MB/s]



--- Training: MobileNetV2 (Fine-tuned) ---
  Epoch 01/15  Train Loss: 0.5906  Train Acc: 0.7800  Val Loss: 0.6803  Val Acc: 0.7562
  Epoch 02/15  Train Loss: 0.4706  Train Acc: 0.8252  Val Loss: 0.6768  Val Acc: 0.7863
  Epoch 03/15  Train Loss: 0.4265  Train Acc: 0.8363  Val Loss: 0.6126  Val Acc: 0.8000
  Epoch 04/15  Train Loss: 0.3987  Train Acc: 0.8514  Val Loss: 0.5147  Val Acc: 0.8263
  Epoch 05/15  Train Loss: 0.3695  Train Acc: 0.8595  Val Loss: 0.5389  Val Acc: 0.8306
  Epoch 06/15  Train Loss: 0.3433  Train Acc: 0.8645  Val Loss: 0.5534  Val Acc: 0.8319
  Epoch 07/15  Train Loss: 0.3358  Train Acc: 0.8682  Val Loss: 0.5114  Val Acc: 0.8538
  Epoch 08/15  Train Loss: 0.3245  Train Acc: 0.8688  Val Loss: 0.4632  Val Acc: 0.8531
  Epoch 09/15  Train Loss: 0.3096  Train Acc: 0.8784  Val Loss: 0.5559  Val Acc: 0.8519
  Epoch 10/15  Train Loss: 0.3217  Train Acc: 0.8775  Val Loss: 0.5363  Val Acc: 0.8569
  Epoch 11/15  Train Loss: 0.2884  Train Acc: 0.8882  Val Loss: 0.5504  Val 

#### Evaluate and Compare All 3 models

In [27]:
criterion = nn.CrossEntropyLoss()

# Get predictions for all 3 models
_, acc1, preds1, labels1 = evaluate(model1, test_loader,    criterion)
_, acc2, preds2, labels2 = evaluate(model2, test_loader,    criterion)
_, acc3, preds3, labels3 = evaluate(model3, pt_test_loader, criterion)

model_names = ['Baseline CNN', 'Improved CNN', 'MobileNetV2']
all_preds   = [preds1,  preds2,  preds3]
all_labels  = [labels1, labels2, labels3]

# Print comparison table
print(f'\n{"Model":<20} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1 Score":>10}')
print('=' * 65)

results = {}
for name, preds, labels in zip(model_names, all_preds, all_labels):
    acc  = np.mean(np.array(preds) == np.array(labels))
    prec = precision_score(labels, preds, average='weighted', zero_division=0)
    rec  = recall_score(   labels, preds, average='weighted', zero_division=0)
    f1   = f1_score(       labels, preds, average='weighted', zero_division=0)
    results[name] = dict(accuracy=acc, precision=prec, recall=rec, f1=f1)
    print(f'{name:<20} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}')

best_model_name = max(results, key=lambda k: results[k]['f1'])
print(f'\n Best Model: {best_model_name}')

NameError: name 'model1' is not defined

#### Plot Training Curves

In [ ]:
histories = [history1, history2, history3]
colors    = ['steelblue', 'darkorange', 'green']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, name, color in zip(histories, model_names, colors):
    # Loss chart
    axes[0].plot(hist['train_loss'], '--', color=color, alpha=0.5, label=f'{name} Train')
    axes[0].plot(hist['val_loss'],   '-',  color=color,             label=f'{name} Val')
    # Accuracy chart
    axes[1].plot(hist['train_acc'], '--', color=color, alpha=0.5)
    axes[1].plot(hist['val_acc'],   '-',  color=color, label=f'{name} Val')

axes[0].set_title('Loss over Epochs',     fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=7)

axes[1].set_title('Accuracy over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=7)

plt.suptitle('Training Curves – All 3 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()
print(' Saved: training_curves.png')

#### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name, preds, labels in zip(axes, model_names, all_preds, all_labels):
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Confusion Matrices – All 3 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png')
plt.show()
print(' Saved: confusion_matrices.png')

#### Metrics Bar Chart (Side-by-Side Comparison)

In [ ]:
metrics_list = ['accuracy', 'precision', 'recall', 'f1']
x     = np.arange(len(metrics_list))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

for i, (name, color) in enumerate(zip(model_names, colors)):
    values = [results[name][m] for m in metrics_list]
    bars   = ax.bar(x + i * width, values, width,
                    label=name, color=color, alpha=0.85)
    # Add value on top of each bar
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1 Score'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison – All Metrics', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('metrics_comparison.png')
plt.show()
print(' Saved: metrics_comparison.png')

#### Detailed Report form Best Model

In [ ]:
best_idx = model_names.index(best_model_name)
print(f'Detailed Classification Report – {best_model_name}')
print('=' * 55)
print(classification_report(
    all_labels[best_idx],
    all_preds[best_idx],
    target_names=CLASS_NAMES
))

#### Deploy: Live Classification UI with Gradio

In [ ]:
# Install gradio once: !pip install gradio
import gradio as gr
from PIL import Image as PILImage

# Use whichever model performed best
BEST_MODEL = model3    # change to model1 or model2 if needed
BEST_MODEL.eval()

# Same transform used during MobileNet testing
ui_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

def predict(image):
    """Takes a PIL image, returns a dictionary of class probabilities."""
    img_tensor = ui_transform(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = BEST_MODEL(img_tensor)
        probs  = torch.softmax(output, dim=1).squeeze().cpu().numpy()
    # Return {class_name: probability} for Gradio to display
    return {CLASS_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)}


# Build the Gradio interface
demo = gr.Interface(
    fn          = predict,
    inputs      = gr.Image(type='pil', label='Upload Brain MRI Image'),
    outputs     = gr.Label(num_top_classes=4, label='Prediction'),
    title       = ' Brain Tumor Classifier',
    description = (
        'Upload a brain MRI image.\n'
        'The model will classify it into one of 4 types:\n'
        'Glioma | Meningioma | No Tumor | Pituitary'
    ),
    flagging_mode = 'never',
)

demo.launch(share=True)   # share=True creates a public link     